## Extracting Checkpoint, Bronze, Silver containers URLs

In [0]:
# Configuration
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("bronze_schema", "bronze", ["bronze", "gabrielajaniszews786_bronze"], "Bronze schema")
dbutils.widgets.combobox("silver_schema", "silver", ["silver", "gabrielajaniszews786_silver"], "Silver schema")
CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

BRONZE_PRICES = f"{CATALOG}.{BRONZE_SCHEMA}.entsoe_prices"

In [0]:
# Creating Silver Schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

In [0]:
# Checking the Bronze layer contents

display(spark.sql(f"DESCRIBE {BRONZE_PRICES}"))
display(spark.sql(f"SELECT * FROM {BRONZE_PRICES} LIMIT 20"))
display(spark.sql(f"SELECT count(*) AS rows, COUNT(DISTINCT timestamp_utc, country) AS unique FROM {BRONZE_PRICES}"))

In [0]:
display(spark.sql(f"""
    SELECT bidding_zone, 
    COUNT(DISTINCT timestamp_utc) as unique_timestamp_utc
    FROM {BRONZE_PRICES}
    GROUP BY bidding_zone"""))

In [0]:
# Confirm duplication: total rows vs. distinct business key
display(spark.sql(f"""
    SELECT COUNT(*)                                     AS total_rows,
           COUNT(DISTINCT timestamp_utc, bidding_zone)  AS unique_key,
           COUNT(*) - COUNT(DISTINCT timestamp_utc, bidding_zone) AS duplicate_rows
    FROM {BRONZE_PRICES}
"""))

In [0]:
# Create the silver fact table with an explicit, enforced schema.
# NOT NULL on key columns is enforced by Delta (real schema enforcement).
# PRIMARY KEY is informational only in Databricks - dedup is done by MERGE, not by this constraint.

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.prices (
  bidding_zone          STRING      NOT NULL,
  timestamp_utc         TIMESTAMP   NOT NULL,
  price                 DECIMAL(10,2),
  currency              STRING,
  unit                  STRING,
  source_file           STRING,
  ingestion_ts          TIMESTAMP,
  silver_processed_ts   TIMESTAMP,
  CONSTRAINT pk_prices PRIMARY KEY (bidding_zone, timestamp_utc)
)
USING DELTA
""")




## Cleaning the prices data

In [0]:
from pyspark.sql import functions as F, Window

# Reading the bronze source
bronze = spark.table(BRONZE_PRICES)

# Deduplication rule: within each business key kepp the most recently ingested row
key_window = Window.partitionBy("bidding_zone", "timestamp_utc").orderBy(F.col("ingestion_ts").desc())

clean = (bronze
         # --- data quality filters that should not reach silver ---
         .filter(F.col("bidding_zone").isNotNull()) # key part must exist
         .filter(F.col("price").isNotNull()) # price is required
         .withColumn("timestamp_utc", F.col("timestamp_utc").cast("timestamp")) # string > timestamp conversion
         # --- typing to match the silver schema established above ---
         .withColumn("price", F.col("price").cast("decimal(10,2)"))
         # --- keep only the latest row per key ---
         .withColumn("row_num", F.row_number().over(key_window))
         .filter(F.col("row_num") == 1)
         # --- silver columns in the same order ---
         .selectExpr(
            "bidding_zone",
            "timestamp_utc",
            "price",
            "currency",
            "unit",
            "source_file",
            "ingestion_ts",
            "current_timestamp() AS silver_processed_ts"        # when silver processed this row
    ))

clean.createOrReplaceTempView("silver_updates")

print("Rows after cleaning and deduplication:", clean.count())

## Upserting into Silver

In [0]:
# First run: nothing MATCHED -> INSERT.
# Re-run: everything is MATCHED -> UPDATE (idempotent, no duplicates).

spark.sql(f"""
    MERGE INTO {CATALOG}.{SILVER_SCHEMA}.prices AS t
    USING silver_updates AS s
      ON t.bidding_zone = s.bidding_zone
     AND t.timestamp_utc = s.timestamp_utc
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
# Verification
display(spark.sql(f"""
    SELECT COUNT(*)                                     AS rows_in_silver,
           COUNT(DISTINCT bidding_zone, timestamp_utc)  AS unique_key
    FROM {CATALOG}.{SILVER_SCHEMA}.prices
"""))

## Quality Rules

In [0]:
# Checking the real price range before setting bounds
display(spark.sql(f"""
    SELECT MIN(price) AS min_price,
           MAX(price) AS max_price
    FROM {CATALOG}.{SILVER_SCHEMA}.prices
"""))

In [0]:
# Enforced data quality rule: Delta validates existing rows and recjects future writes that violate the price range rule

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.prices
    ADD CONSTRAINT price_range CHECK (price BETWEEN -1000 AND 10000)
""")

In [0]:
# Testing the constraint

try:
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SILVER_SCHEMA}.prices
        VALUES ('PL', TIMESTAMP'2099-01-01T00:00:00', 999999.00,
                'EUR', 'MWH', 'demo_bad_row.json',
                current_timestamp(), current_timestamp())
    """)
    print("Insert succeeded - constraint NOT working (not good!)")
except Exception as e:
    print(f"Insert failed - constraint working (good!):\n")
    print(str(e))

In [0]:
# Checking if the number of rows is still correct
display(spark.sql(f"SELECT COUNT(*) AS rows_in_silver FROM {CATALOG}.{SILVER_SCHEMA}.prices"))